In [ ]:
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
import csv
from datetime import datetime, timezone
import time

In [ ]:
# 설정
SCOPES = ["https://www.googleapis.com/auth/youtube.readonly"]
OUTPUT_FILE = "youtube_lol_playlists.csv"
MAX_COMMENTS_PER_VIDEO = 500

# OAuth 2.0 인증 (자신 계정)
flow = InstalledAppFlow.from_client_secrets_file(
    "client_secrets.json", SCOPES
)
credentials = flow.run_console()
youtube = build("youtube", "v3", credentials=credentials)

In [ ]:
# 재생목록 가져오기
def fetch_playlists():
    playlists = []
    next_page_token = None
    while True:
        res = youtube.playlists().list(
            part="id,snippet",
            mine=True,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        for pl in res.get("items", []):
            playlists.append({
                "playlist_id": pl["id"],
                "title": pl["snippet"]["title"]
            })
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return playlists

In [ ]:
# 재생목록 안 영상 가져오기
def fetch_playlist_videos(playlist_id):
    videos = []
    next_page_token = None
    while True:
        res = youtube.playlistItems().list(
            part="snippet",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        for item in res.get("items", []):
            videos.append({
                "video_id": item["snippet"]["resourceId"]["videoId"],
                "title": item["snippet"]["title"],
                "description": item["snippet"]["description"],
                "publish_time": item["snippet"]["publishedAt"]
            })
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return videos

In [ ]:
# 영상 통계 조회
# 조회수, 좋아요 수, 댓글 수 통계 정보
def fetch_video_stats(video_id):
    res = youtube.videos().list(
        part="statistics",
        id=video_id
    ).execute()
    stats = res.get("items", [])
    if stats:
        stat = stats[0]["statistics"]
        return {
            "viewCount": int(stat.get("viewCount", 0)),
            "likeCount": int(stat.get("likeCount", 0)),
            "commentCount": int(stat.get("commentCount", 0))
        }
    else:
        return {"viewCount": 0, "likeCount": 0, "commentCount": 0}

In [ ]:
# 댓글 수집
def fetch_comments(video_id):
    comments = []
    next_page_token = None
    while True:
        if len(comments) >= MAX_COMMENTS_PER_VIDEO:
            break
        res = youtube.commentThreads().list(
            part="snippet,replies",
            videoId=video_id,
            maxResults=100,
            textFormat="plainText",
            pageToken=next_page_token
        ).execute()
        for thread in res.get("items", []):
            if len(comments) >= MAX_COMMENTS_PER_VIDEO:
                break
            top_comment = thread["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
            comments.append(top_comment)
            if "replies" in thread:
                for reply in thread["replies"]["comments"]:
                    if len(comments) >= MAX_COMMENTS_PER_VIDEO:
                        break
                    comments.append(reply["snippet"]["textDisplay"])
        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break
    return comments

In [ ]:
# 영상 + 댓글 처리
def process_video(video, playlist_title):
    video_id = video["video_id"]
    publish_utc = datetime.fromisoformat(video["publish_time"].replace("Z","+00:00")).strftime("%Y-%m-%d %H:%M:%S")
    comments = fetch_comments(video_id)
    stats = fetch_video_stats(video_id)
    rows = []

    if not comments:
        rows.append([
            playlist_title,
            video_id,
            video["title"],
            video["description"],
            publish_utc,
            stats["viewCount"],
            stats["likeCount"],
            stats["commentCount"],
            ""
        ])
    else:
        for c in comments:
            rows.append([
                playlist_title,
                video_id,
                video["title"],
                video["description"],
                publish_utc,
                stats["viewCount"],
                stats["likeCount"],
                stats["commentCount"],
                c
            ])
    return rows

In [ ]:
# 메인 수집
def collect_youtube_playlists():
    total_processed = 0
    start_time = time.time()

    playlists = fetch_playlists()
    print(f"총 {len(playlists)}개의 재생목록 발견")

    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "playlist_title",
            "video_id",
            "title",
            "description",
            "publish_time_utc",
            "viewCount",
            "likeCount",
            "commentCount",
            "comment"
        ])

        for pl in playlists:
            playlist_title = pl["title"]
            print(f"\n===== 재생목록 수집 시작: {playlist_title} =====")
            videos = fetch_playlist_videos(pl["playlist_id"])
            print(f"총 {len(videos)}개의 영상 발견")

            for video in videos:
                rows = process_video(video, playlist_title)
                for row in rows:
                    writer.writerow(row)
                total_processed += 1

                # 진행률 및 ETA
                progress_ratio = total_processed / max(len(videos),1)
                elapsed_time = time.time() - start_time
                avg_time = elapsed_time / max(total_processed,1)
                remaining_time = avg_time * (len(videos) - total_processed)

                if total_processed % 5 == 0:
                    print(
                        f"[{playlist_title}] 저장 중 영상 ID: {rows[0][1]} | "
                        f"게시글 UTC: {rows[0][4]} | "
                        f"진행률: {progress_ratio*100:.2f}% | "
                        f"총 처리: {total_processed}개 | "
                        f"평균: {avg_time:.2f}초/개 | "
                        f"예상 남은 시간: {remaining_time/60:.1f}분"
                    )

            print(f"===== 재생목록 수집 완료: {playlist_title} =====")

    print("\n===== 전체 YouTube 재생목록 수집 완료 =====")

In [ ]:
# 실행
collect_youtube_playlists()